# GUS04E — Validation & Diagnostics

Comprehensive quality assessment for all E_ estimation pipelines:
- **Consistency diagnostics** (item 30): non-negativity, marginal/hierarchical consistency, population match, temporal smoothness, sub-division consistency
- **Estimation confidence scoring** (item 31): per-territory confidence based on anchor availability, data coverage, population size
- **Leave-one-out cross-validation** (item 29): holdout census year, compare predicted vs observed

**Prerequisites**: Run GUS04D (all 8 estimation pipelines completed) on `geoteryt_O.pkl`.

In [1]:
# ── Cell 1: Imports & load database ──
import sys, os, time
import numpy as np
import pandas as pd

REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, os.path.join(REPO, 'Code', 'tools'))
DATA_ROOT = os.path.join(REPO, '..', '..', 'Data', 'Geospatial')

from geoTERYT_db import (
    load_complete_database, LEVEL_GMINA, LEVEL_VOIVODESHIP, LEVEL_POWIAT,
)

db_path = os.path.join(DATA_ROOT, 'geoteryt_O.pkl')
print(f"Loading database from {db_path}…")
t0 = time.time()
db = load_complete_database(db_path)
print(f"Loaded in {time.time()-t0:.1f}s — {len(db._records)} records")

Loading database from /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/local_repo/LRDWI-Paper/../../Data/Geospatial/geoteryt_O.pkl…
Loading complete database from /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/local_repo/LRDWI-Paper/../../Data/Geospatial/geoteryt_O.pkl...
  Database version: 4.3
  ✓ Restored old voivodships: 49 rows
  ✓ Restored geometry store: 13,287 unique geometries
  ✓ Restored geometry data for years: [2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2015, 2016, 2017, 2018, 2021, 2022, 2023]
  ✓ Loaded 4612 records
  ✓ Year range: 1999 - 2024
  ✓ Records with geometry: 3661
  ✓ Records with old_woj: 4104
  ✓ Records with data: 4584
  ✓ Records with cross tables: 4584
  ✓ Records with population data: 4582
  ✓ Records with pop_class: 3411
Loaded in 350.2s — 4612 records


In [7]:
# ── Cell 2: Initialize DemographicEstimator ──
import importlib
import demographic_estimator
importlib.reload(demographic_estimator)
from demographic_estimator import (
    DemographicEstimator, E_SUBJECT_NAMES,
    PREDICTION_2000_RANGE, PREDICTION_1990_RANGE,
)

est = DemographicEstimator(db, verbose=True)
print(repr(est))

# List all E_ subjects present in the database
all_e_sids = sorted(E_SUBJECT_NAMES.values())
print(f"\nE_ subjects defined: {len(all_e_sids)}")
for e_sid in all_e_sids:
    n = sum(1 for r in db._records.values()
            if e_sid in r.cross_tables and r.cross_tables[e_sid].years_with_data)
    print(f"  {e_sid}: {n} records with data")

DemographicEstimator initialised  (Gurobi=YES, IPFN=YES)
DemographicEstimator(completed=0/9, Gurobi=YES)

E_ subjects defined: 9
  E_age_educ_2000: 0 records with data
  E_age_sex_1990: 2671 records with data
  E_age_sex_2000: 3069 records with data
  E_educ_1990: 2928 records with data
  E_educ_2000: 2954 records with data
  E_educ_sex_1990: 2928 records with data
  E_educ_sex_2000: 2954 records with data
  E_hh_size_1990: 2928 records with data
  E_hh_size_2000: 2954 records with data


In [3]:
# ── Cell 2b: Run all 8 estimation pipelines ──
# The E_ results live in memory only — must re-run after fresh DB load.

pipelines = [
    ('age_sex', '2000'),
    ('age_sex', '1990'),
    ('educ', '2000'),
    ('educ', '1990'),
    ('educ_sex', '2000'),
    ('educ_sex', '1990'),
    ('hh_size', '2000'),
    ('hh_size', '1990'),
]

t_total = time.time()
for var_type, pred_section in pipelines:
    t0 = time.time()
    print(f"\n{'─'*50}")
    est.run_pipeline(var_type, pred_section)
    print(f"  → {time.time()-t0:.1f}s")

print(f"\n{'='*50}")
print(f"All pipelines completed in {time.time()-t_total:.1f}s")
print(repr(est))


──────────────────────────────────────────────────

  PIPELINE: age_sex / Prediction2000
  Output subject: E_age_sex_2000
  Source: M_age_sex  shape=(16, 3)
  Gminas total: 2671, with M_age_sex: 2671
  Layer 1: generating seeds (log-linear interpolation)…
    Seeds generated: 2671/2671 units (skipped 0)
    2000: 2490 obs + 0 est
    2005: 2479 obs + 0 est
    2010: 2480 obs + 0 est
    2015: 2479 obs + 0 est
    2020: 2478 obs + 0 est
    2025: 0 obs + 2478 est
  Aggregating to powiat and voivodeship levels…
    Aggregated: 10229 powiat-years, 432 voiv-years
  Summary: 64486 observed + 2478 estimated cell-years stored
  ✓  E_age_sex_2000 complete
  → 15.8s

──────────────────────────────────────────────────

  PIPELINE: age_sex / Prediction1990
  Output subject: E_age_sex_1990
  Source: M_age_sex  shape=(16, 3)
  Phase A: constructing 1988 gmina age×sex via IPF…
    1988 IPF: 2478 OK, 193 skipped
  Phase B: building seeds (log-linear interpolation)…
    Seeds: 2671 gminas
  Phase C: 

## 1. Consistency Diagnostics (Item 30)

For each E_ subject, run `validate_results()` checking:
1. Non-negativity (all cells ≥ 0)
2. Marginal consistency (ogółem = sum of parts)
3. Hierarchical consistency (children sum = parent)
4. Population match (grand total ≈ record.pop)
5. Temporal smoothness (no >20% year-over-year jumps)
6. Sub-division consistency (rodz-3 = rodz-4 + rodz-5)

In [8]:
# ── Cell 3: Run consistency diagnostics for all E_ subjects ──

all_diag = {}
for e_sid in all_e_sids:
    # Check if any records have this subject
    n = sum(1 for r in db._records.values()
            if e_sid in r.cross_tables and r.cross_tables[e_sid].years_with_data)
    if n == 0:
        print(f"\n⊘ {e_sid}: no data — skipping")
        continue
    print(f"\n{'='*60}")
    print(f"Validating {e_sid} ({n} records)…")
    print(f"{'='*60}")
    t0 = time.time()
    diag_df = est.validate_results(e_sid)
    all_diag[e_sid] = diag_df
    print(f"  Time: {time.time()-t0:.1f}s")
    if diag_df.empty:
        print(f"  ✓ No issues found!")
    else:
        print(f"  Issues found: {len(diag_df)}")
        # Summary by check type
        summary = diag_df.groupby(['check', 'status']).size().reset_index(name='count')
        print(summary.to_string(index=False))


⊘ E_age_educ_2000: no data — skipping

Validating E_age_sex_1990 (2671 records)…
  Validating E_age_sex_1990: 2671 records, shape=(16, 3)
    [1] Non-negativity: 0 failures
    [2] Marginal consistency: 0 failures (tol=1.0)
    [3] Hierarchical consistency: 0 failures / 0 checked
    [4] Population match: 0 failures / 23816 checked (tol=0.1%)
    [5] Temporal smoothness: 5463 warnings (threshold=20%)
    [6] Sub-division consistency: 0 failures / 0 checked
  Validation complete: 5463 issues found
  Time: 2.7s
  Issues found: 5463
              check status  count
temporal_smoothness   WARN   5463

Validating E_age_sex_2000 (3069 records)…
  Validating E_age_sex_2000: 3069 records, shape=(16, 3)
    [1] Non-negativity: 6 failures
    [2] Marginal consistency: 0 failures (tol=1.0)
    [3] Hierarchical consistency: 0 failures / 10229 checked
    [4] Population match: 0 failures / 64486 checked (tol=0.1%)
    [5] Temporal smoothness: 7079 warnings (threshold=20%)
    [6] Sub-division cons

In [9]:
# ── Cell 4: Diagnostic summary table ──

summary_rows = []
for e_sid, diag_df in all_diag.items():
    if diag_df.empty:
        summary_rows.append({
            'subject': e_sid,
            'total_issues': 0,
            'failures': 0,
            'warnings': 0,
            'checks_passed': 'ALL',
        })
    else:
        n_fail = (diag_df['status'] == 'FAIL').sum()
        n_warn = (diag_df['status'] == 'WARN').sum()
        failed_checks = diag_df.loc[diag_df['status']=='FAIL', 'check'].unique()
        summary_rows.append({
            'subject': e_sid,
            'total_issues': len(diag_df),
            'failures': n_fail,
            'warnings': n_warn,
            'checks_passed': ', '.join(sorted(failed_checks)) if len(failed_checks) else 'ALL',
        })

summary_df = pd.DataFrame(summary_rows)
print("\n" + "="*80)
print("DIAGNOSTIC SUMMARY")
print("="*80)
print(summary_df.to_string(index=False))


DIAGNOSTIC SUMMARY
        subject  total_issues  failures  warnings        checks_passed
 E_age_sex_1990          5463         0      5463                  ALL
 E_age_sex_2000          7085         6      7079       non_negativity
    E_educ_1990          8348      2334      6014 marginal_consistency
    E_educ_2000          4403         0      4403                  ALL
E_educ_sex_1990          9958      2334      7624 marginal_consistency
E_educ_sex_2000          4822         0      4822                  ALL
 E_hh_size_1990             0         0         0                  ALL
 E_hh_size_2000            26         0        26                  ALL


In [10]:
# ── Cell 5: Deep-dive — population match failures ──

for e_sid, diag_df in all_diag.items():
    pop_fails = diag_df[diag_df['check'] == 'population_match']
    if pop_fails.empty:
        continue
    print(f"\n{'='*60}")
    print(f"{e_sid}: {len(pop_fails)} population match failures")
    print(f"{'='*60}")
    # Show worst 20
    # Extract error % from detail string
    pop_fails = pop_fails.copy()
    pop_fails['err_pct'] = pop_fails['detail'].str.extract(r'err=([0-9.]+)%').astype(float)
    worst = pop_fails.nlargest(20, 'err_pct')
    for _, row in worst.iterrows():
        print(f"  {row['teryt_id']} {row['name']:30s} yr={row['year']}  {row['detail']}")

In [11]:
# ── Cell 6: Deep-dive — hierarchical consistency failures ──

for e_sid, diag_df in all_diag.items():
    hier_fails = diag_df[diag_df['check'] == 'hierarchical_consistency']
    if hier_fails.empty:
        continue
    print(f"\n{'='*60}")
    print(f"{e_sid}: {len(hier_fails)} hierarchical consistency failures")
    print(f"{'='*60}")
    hier_fails = hier_fails.copy()
    hier_fails['pct_err'] = hier_fails['detail'].str.extract(r'\(([0-9.]+)%').astype(float)
    worst = hier_fails.nlargest(20, 'pct_err')
    for _, row in worst.iterrows():
        print(f"  {row['teryt_id']} {row['name']:30s} yr={row['year']}  {row['detail']}")

In [12]:
# ── Cell 7: Deep-dive — temporal smoothness warnings ──

for e_sid, diag_df in all_diag.items():
    smooth_warns = diag_df[diag_df['check'] == 'temporal_smoothness']
    if smooth_warns.empty:
        print(f"{e_sid}: ✓ No temporal smoothness warnings")
        continue
    print(f"\n{'='*60}")
    print(f"{e_sid}: {len(smooth_warns)} temporal smoothness warnings")
    print(f"{'='*60}")
    smooth_warns = smooth_warns.copy()
    smooth_warns['max_rel'] = smooth_warns['detail'].str.extract(r'max_rel_change=([0-9.]+)').astype(float)
    # Distribution
    print(f"  max_rel_change distribution:")
    print(f"    mean:   {smooth_warns['max_rel'].mean():.3f}")
    print(f"    median: {smooth_warns['max_rel'].median():.3f}")
    print(f"    P95:    {smooth_warns['max_rel'].quantile(0.95):.3f}")
    print(f"    max:    {smooth_warns['max_rel'].max():.3f}")
    # Top 10 worst
    worst = smooth_warns.nlargest(10, 'max_rel')
    print(f"  Top 10 worst:")
    for _, row in worst.iterrows():
        print(f"    {row['teryt_id']} {row['name']:30s} yr={row['year']}  {row['detail']}")


E_age_sex_1990: 5463 temporal smoothness warnings
  max_rel_change distribution:
    mean:   0.260
    median: 0.233
    P95:    0.413
    max:    15.982
  Top 10 worst:
    1412031 Wesoła                         yr=2002  max_rel_change=15.982 (2001→2002)
    2408052 Wyry                           yr=1999  max_rel_change=1.406 (1998→1999)
    1008072 Pabianice                      yr=1997  max_rel_change=1.269 (1996→1997)
    1207122 Tymbark                        yr=1997  max_rel_change=1.084 (1996→1997)
    1431021 Warszawa-Białołęka             yr=1999  max_rel_change=0.991 (1998→1999)
    1431171 Warszawa-Wilanów               yr=1999  max_rel_change=0.933 (1998→1999)
    2414042 Bojszowy                       yr=1999  max_rel_change=0.921 (1998→1999)
    2410022 Kobiór                         yr=1999  max_rel_change=0.842 (1998→1999)
    0208092 Lewin Kłodzki                  yr=1989  max_rel_change=0.823 (1988→1989)
    0601152 Sosnówka                       yr=1989  max_rel_cha

## 2. Estimation Confidence Scores (Item 31)

For each gmina and year, compute a confidence score (0–100) based on:
- Number of census anchors (0–4) → 30%
- Observed data coverage → 25%
- Distance to nearest anchor → 20%
- Population size (log-scaled) → 15%
- Historical code usage → 10%

In [13]:
# ── Cell 8: Compute confidence scores for all E_ subjects ──

all_confidence = {}
for e_sid in all_e_sids:
    n = sum(1 for r in db._records.values()
            if e_sid in r.cross_tables and r.cross_tables[e_sid].years_with_data)
    if n == 0:
        print(f"⊘ {e_sid}: no data — skipping")
        continue
    print(f"\n{'='*60}")
    print(f"Confidence scores: {e_sid}")
    print(f"{'='*60}")
    conf_df = est.compute_confidence_scores(e_sid)
    all_confidence[e_sid] = conf_df

⊘ E_age_educ_2000: no data — skipping

Confidence scores: E_age_sex_1990
  Confidence scores for E_age_sex_1990: 2671 gminas, 45407 gmina-years
    Mean confidence: 47.4
    Median: 46.5
    Min: 9.8, Max: 66.4

Confidence scores: E_age_sex_2000
  Confidence scores for E_age_sex_2000: 2671 gminas, 66964 gmina-years
    Mean confidence: 85.2
    Median: 84.9
    Min: 19.5, Max: 98.6

Confidence scores: E_educ_1990
  Confidence scores for E_educ_1990: 2531 gminas, 43027 gmina-years
    Mean confidence: 52.6
    Median: 52.7
    Min: 25.8, Max: 67.6

Confidence scores: E_educ_2000
  Confidence scores for E_educ_2000: 2557 gminas, 69039 gmina-years
    Mean confidence: 50.0
    Median: 50.5
    Min: 25.1, Max: 66.4

Confidence scores: E_educ_sex_1990
  Confidence scores for E_educ_sex_1990: 2531 gminas, 43027 gmina-years
    Mean confidence: 37.2
    Median: 36.3
    Min: 14.3, Max: 56.1

Confidence scores: E_educ_sex_2000
  Confidence scores for E_educ_sex_2000: 2557 gminas, 69039 gmina-y

In [14]:
# ── Cell 9: Confidence summary across subjects ──

conf_rows = []
for e_sid, conf_df in all_confidence.items():
    n_gminas = conf_df['teryt_id'].nunique()
    avg_conf = conf_df.groupby('teryt_id')['confidence'].mean()
    conf_rows.append({
        'subject': e_sid,
        'n_gminas': n_gminas,
        'mean_conf': f"{avg_conf.mean():.1f}",
        'median_conf': f"{avg_conf.median():.1f}",
        'min_conf': f"{avg_conf.min():.1f}",
        'max_conf': f"{avg_conf.max():.1f}",
        'pct_below_40': f"{(avg_conf < 40).mean()*100:.1f}%",
    })

csummary = pd.DataFrame(conf_rows)
print("\n" + "="*80)
print("CONFIDENCE SCORE SUMMARY (per-gmina average across years)")
print("="*80)
print(csummary.to_string(index=False))


CONFIDENCE SCORE SUMMARY (per-gmina average across years)
        subject  n_gminas mean_conf median_conf min_conf max_conf pct_below_40
 E_age_sex_1990      2671      47.4        47.8     11.3     55.6         0.4%
 E_age_sex_2000      2671      84.6        85.2     19.6     92.9         0.4%
    E_educ_1990      2531      52.6        53.1     30.2     60.9         3.6%
    E_educ_2000      2557      50.0        51.0     30.0     58.7         6.2%
E_educ_sex_1990      2531      37.2        37.6     15.9     45.3        93.1%
E_educ_sex_2000      2557      50.0        51.0     30.0     58.7         6.2%
 E_hh_size_1990      2531      52.6        53.1     30.2     60.9         3.6%
 E_hh_size_2000      2557      50.0        51.0     30.0     58.7         6.2%


In [15]:
# ── Cell 10: Lowest-confidence gminas (potential problem areas) ──

# Pick the most important subject (E_age_sex_2000) for detailed analysis
focus_sid = 'E_age_sex_2000'
if focus_sid in all_confidence:
    conf_df = all_confidence[focus_sid]
    avg_conf = conf_df.groupby(['teryt_id', 'name'])['confidence'].mean().reset_index()
    avg_conf = avg_conf.sort_values('confidence')
    
    print(f"\n{'='*60}")
    print(f"Lowest-confidence gminas for {focus_sid}")
    print(f"{'='*60}")
    print(f"\nBottom 30 gminas (mean confidence across years):")
    for _, row in avg_conf.head(30).iterrows():
        # Get detail for this gmina
        g_data = conf_df[conf_df['teryt_id'] == row['teryt_id']].iloc[0]
        print(f"  {row['teryt_id']} {row['name']:32s}  conf={row['confidence']:.1f}  "
              f"anchors={g_data['n_anchors']}  obs_yrs={g_data['n_observed_years']}  "
              f"hist={'Y' if g_data['used_historical'] else 'N'}")
    
    # Distribution
    print(f"\nDistribution of mean confidence:")
    for pctile in [5, 10, 25, 50, 75, 90, 95]:
        val = avg_conf['confidence'].quantile(pctile/100)
        print(f"  P{pctile:2d}: {val:.1f}")
else:
    print(f"  {focus_sid} not available — run GUS04D first")


Lowest-confidence gminas for E_age_sex_2000

Bottom 30 gminas (mean confidence across years):
  1431171 Warszawa-Wilanów                  conf=19.6  anchors=0  obs_yrs=3  hist=Y
  1431121 Warszawa-Rembertów                conf=20.4  anchors=0  obs_yrs=3  hist=Y
  1431181 Warszawa-Włochy                   conf=21.3  anchors=0  obs_yrs=3  hist=Y
  1431141 Warszawa-Ursus                    conf=21.5  anchors=0  obs_yrs=3  hist=Y
  1431021 Warszawa-Białołęka                conf=21.8  anchors=0  obs_yrs=3  hist=Y
  1431161 Warszawa-Wawer                    conf=22.1  anchors=0  obs_yrs=3  hist=Y
  1431011 Warszawa-Bemowo                   conf=22.9  anchors=0  obs_yrs=3  hist=Y
  1431131 Warszawa-Targówek                 conf=23.2  anchors=0  obs_yrs=3  hist=Y
  1431151 Warszawa-Ursynów                  conf=23.3  anchors=0  obs_yrs=3  hist=Y
  1431031 Warszawa-Bielany                  conf=23.4  anchors=0  obs_yrs=3  hist=Y
  1431041 Warszawa-Centrum                  conf=31.5  anchors=0 

In [16]:
# ── Cell 11: Confidence by year — temporal pattern ──

if focus_sid in all_confidence:
    conf_df = all_confidence[focus_sid]
    yearly = conf_df.groupby('year')['confidence'].agg(['mean', 'std', 'min', 'max'])
    yearly.columns = ['mean_conf', 'std_conf', 'min_conf', 'max_conf']
    
    print(f"\n{'='*60}")
    print(f"Confidence by year for {focus_sid}")
    print(f"{'='*60}")
    print(yearly.to_string())
    
    print(f"\nNote: Higher confidence near census years (2002, 2011, 2021).")
    print(f"Lower confidence for years far from anchors (1999, 2024-2025).")


Confidence by year for E_age_sex_2000
      mean_conf  std_conf  min_conf  max_conf
year                                         
1999  83.172570  4.624663      19.5      91.1
2000  84.950602  4.720393      19.6      92.9
2001  87.322169  4.849102      19.8      95.3
2002  90.941710  2.263088      46.2      98.6
2003  87.615571  2.144486      60.8      95.3
2004  85.235458  2.130789      61.4      92.9
2005  83.451190  2.120313      61.0      91.1
2006  82.070109  2.065819      60.7      89.8
2007  82.070553  2.061857      60.7      89.8
2008  83.459580  2.061284      62.1      91.1
2009  85.244897  2.060675      63.9      92.9
2010  87.617258  2.107399      65.5      95.3
2011  90.950645  2.108359      68.8      98.6
2012  87.617258  2.107399      65.5      95.3
2013  85.244435  2.063674      63.1      92.9
2014  83.459113  2.064285      61.3      91.1
2015  82.077289  2.033740      60.0      89.8
2016  80.966277  2.033634      58.8      88.6
2017  82.076482  2.040941      60.0      

## 3. Leave-One-Out Cross-Validation (Item 29)

For the age×sex pipeline (best data coverage), hold out one census year and re-estimate.
Compare predicted vs. observed at the holdout year.

**Note:** LOOCV re-runs the full pipeline for each holdout, so this section is computationally intensive.
We test with **Census 2011** (holdout) for E_age_sex_2000 as a representative example.

In [17]:
# ── Cell 12: LOOCV — age_sex_2000, holdout=2011 ──

print("Running leave-one-out cross-validation…")
print("Pipeline: age_sex, Section: 2000, Holdout year: 2011")
print("This re-runs the full age_sex_2000 pipeline without 2011 data.")
print()

t0 = time.time()
loocv_df = est.leave_one_out_cv('age_sex', '2000', holdout_year=2011)
print(f"\nCompleted in {time.time()-t0:.1f}s")

Running leave-one-out cross-validation…
Pipeline: age_sex, Section: 2000, Holdout year: 2011
This re-runs the full age_sex_2000 pipeline without 2011 data.


  LOOCV: E_age_sex_2000, holdout=2011
    source=M_age_sex, shape=(16, 3)
    Observed gminas at 2011: 2659
    Predicted gminas at holdout: 2480
    Evaluated: 2480 gminas
    Mean RMSE: 35.17
    Mean RMSE%: 6.22%
    Mean χ²: 180.29
    Mean pop err%: 3.341%

Completed in 15.4s


In [18]:
# ── Cell 13: LOOCV results analysis ──

if not loocv_df.empty:
    print(f"{'='*60}")
    print(f"LOOCV Results: age_sex_2000, holdout=2011")
    print(f"{'='*60}")
    print(f"Evaluated gminas: {len(loocv_df)}")
    print()
    
    # Overall statistics
    for col in ['cell_rmse', 'cell_rmse_pct', 'chi_sq', 'marginal_err', 'total_pop_err_pct']:
        vals = loocv_df[col]
        print(f"  {col:25s}  mean={vals.mean():10.3f}  "
              f"median={vals.median():10.3f}  "
              f"P95={vals.quantile(0.95):10.3f}  "
              f"max={vals.max():10.3f}")
    
    # Top 10 worst by RMSE%
    print(f"\nTop 10 worst gminas by RMSE%:")
    worst = loocv_df.nlargest(10, 'cell_rmse_pct')
    for _, row in worst.iterrows():
        print(f"  {row['teryt_id']} {row['name']:30s}  "
              f"RMSE={row['cell_rmse']:.1f}  "
              f"RMSE%={row['cell_rmse_pct']:.1f}%  "
              f"χ²={row['chi_sq']:.1f}  "
              f"pop_err={row['total_pop_err_pct']:.2f}%")
else:
    print("No LOOCV results — check pipeline output above.")

LOOCV Results: age_sex_2000, holdout=2011
Evaluated gminas: 2480

  cell_rmse                  mean=    35.168  median=     9.412  P95=    77.268  max= 15981.099
  cell_rmse_pct              mean=     6.223  median=     3.575  P95=    25.514  max=    28.206
  chi_sq                     mean=   180.286  median=     9.820  P95=   473.856  max=108359.873
  marginal_err               mean=   124.643  median=    28.855  P95=   256.812  max= 63366.708
  total_pop_err_pct          mean=     3.341  median=     0.352  P95=    23.818  max=    24.997

Top 10 worst gminas by RMSE%:
  1409012 Chotcza                         RMSE=23.9  RMSE%=28.2%  χ²=150.6  pop_err=23.79%
  1431001 Warszawa                        RMSE=15981.1  RMSE%=28.1%  χ²=108359.9  pop_err=25.00%
  1465011 Warszawa                        RMSE=15981.1  RMSE%=28.1%  χ²=108359.9  pop_err=25.00%
  1404032 Pacyna                          RMSE=35.5  RMSE%=27.9%  χ²=232.7  pop_err=24.26%
  1409062 Solec nad Wisłą                 RMSE=

In [19]:
# ── Cell 14: LOOCV — age_sex_2000, holdout=2002 ──

print("Running LOOCV with holdout=2002…")
t0 = time.time()
loocv_2002_df = est.leave_one_out_cv('age_sex', '2000', holdout_year=2002)
print(f"\nCompleted in {time.time()-t0:.1f}s")

if not loocv_2002_df.empty:
    print(f"\nEvaluated gminas: {len(loocv_2002_df)}")
    for col in ['cell_rmse', 'cell_rmse_pct', 'chi_sq', 'total_pop_err_pct']:
        vals = loocv_2002_df[col]
        print(f"  {col:25s}  mean={vals.mean():10.3f}  "
              f"median={vals.median():10.3f}  "
              f"max={vals.max():10.3f}")
    
    print(f"\nTop 10 worst by RMSE%:")
    worst = loocv_2002_df.nlargest(10, 'cell_rmse_pct')
    for _, row in worst.iterrows():
        print(f"  {row['teryt_id']} {row['name']:30s}  "
              f"RMSE%={row['cell_rmse_pct']:.1f}%  "
              f"pop_err={row['total_pop_err_pct']:.2f}%")

Running LOOCV with holdout=2002…

  LOOCV: E_age_sex_2000, holdout=2002
    source=M_age_sex, shape=(16, 3)
    Observed gminas at 2002: 2658
    Predicted gminas at holdout: 2479
    Evaluated: 2479 gminas
    Mean RMSE: 33.56
    Mean RMSE%: 6.30%
    Mean χ²: 180.31
    Mean pop err%: 3.373%

Completed in 14.8s

Evaluated gminas: 2479
  cell_rmse                  mean=    33.562  median=     9.177  max= 15492.920
  cell_rmse_pct              mean=     6.298  median=     3.603  max=    30.793
  chi_sq                     mean=   180.313  median=     9.516  max=109408.300
  total_pop_err_pct          mean=     3.373  median=     0.290  max=    25.678

Top 10 worst by RMSE%:
  1409012 Chotcza                         RMSE%=30.8%  pop_err=25.10%
  1423042 Odrzywół                        RMSE%=29.0%  pop_err=25.68%
  1426072 Przesmyki                       RMSE%=28.5%  pop_err=24.70%
  1409062 Solec nad Wisłą                 RMSE%=28.5%  pop_err=24.81%
  1429032 Ceranów                   

In [20]:
# ── Cell 15: LOOCV — age_sex_2000, holdout=2021 ──

print("Running LOOCV with holdout=2021…")
t0 = time.time()
loocv_2021_df = est.leave_one_out_cv('age_sex', '2000', holdout_year=2021)
print(f"\nCompleted in {time.time()-t0:.1f}s")

if not loocv_2021_df.empty:
    print(f"\nEvaluated gminas: {len(loocv_2021_df)}")
    for col in ['cell_rmse', 'cell_rmse_pct', 'chi_sq', 'total_pop_err_pct']:
        vals = loocv_2021_df[col]
        print(f"  {col:25s}  mean={vals.mean():10.3f}  "
              f"median={vals.median():10.3f}  "
              f"max={vals.max():10.3f}")
    
    print(f"\nTop 10 worst by RMSE%:")
    worst = loocv_2021_df.nlargest(10, 'cell_rmse_pct')
    for _, row in worst.iterrows():
        print(f"  {row['teryt_id']} {row['name']:30s}  "
              f"RMSE%={row['cell_rmse_pct']:.1f}%  "
              f"pop_err={row['total_pop_err_pct']:.2f}%")

Running LOOCV with holdout=2021…

  LOOCV: E_age_sex_2000, holdout=2021
    source=M_age_sex, shape=(16, 3)
    Observed gminas at 2021: 2654
    Predicted gminas at holdout: 2478
    Evaluated: 2478 gminas
    Mean RMSE: 39.04
    Mean RMSE%: 6.67%
    Mean χ²: 204.91
    Mean pop err%: 3.704%

Completed in 15.5s

Evaluated gminas: 2478
  cell_rmse                  mean=    39.036  median=     9.799  max= 17605.417
  cell_rmse_pct              mean=     6.667  median=     3.830  max=    30.783
  chi_sq                     mean=   204.912  median=    11.078  max=120274.473
  total_pop_err_pct          mean=     3.704  median=     0.492  max=    26.952

Top 10 worst by RMSE%:
  1426022 Korczew                         RMSE%=30.8%  pop_err=26.01%
  1429032 Ceranów                         RMSE%=30.2%  pop_err=25.80%
  1409012 Chotcza                         RMSE%=30.0%  pop_err=26.11%
  1416062 Nur                             RMSE%=29.6%  pop_err=25.32%
  1426072 Przesmyki                 

In [21]:
# ── Cell 16: LOOCV comparison across holdout years ──

loocv_summary = []
for holdout, df in [('2002', loocv_2002_df), ('2011', loocv_df), ('2021', loocv_2021_df)]:
    if df is not None and not df.empty:
        loocv_summary.append({
            'holdout_year': holdout,
            'n_gminas': len(df),
            'mean_rmse': f"{df['cell_rmse'].mean():.2f}",
            'mean_rmse_pct': f"{df['cell_rmse_pct'].mean():.1f}%",
            'median_rmse_pct': f"{df['cell_rmse_pct'].median():.1f}%",
            'mean_chi_sq': f"{df['chi_sq'].mean():.1f}",
            'mean_pop_err': f"{df['total_pop_err_pct'].mean():.3f}%",
        })

if loocv_summary:
    comp_df = pd.DataFrame(loocv_summary)
    print("\n" + "="*80)
    print("LOOCV COMPARISON ACROSS HOLDOUT YEARS (age_sex_2000)")
    print("="*80)
    print(comp_df.to_string(index=False))
    print("\nInterpretation:")
    print("  - Lower RMSE% = better prediction accuracy")
    print("  - Edge years (2002, 2021) may show worse performance")
    print("    (fewer bracketing anchors for extrapolation)")
    print("  - 2011 (interior) typically shows best performance")
    print("    (well-bounded by 2002 and 2021 anchors)")


LOOCV COMPARISON ACROSS HOLDOUT YEARS (age_sex_2000)
holdout_year  n_gminas mean_rmse mean_rmse_pct median_rmse_pct mean_chi_sq mean_pop_err
        2002      2479     33.56          6.3%            3.6%       180.3       3.373%
        2011      2480     35.17          6.2%            3.6%       180.3       3.341%
        2021      2478     39.04          6.7%            3.8%       204.9       3.704%

Interpretation:
  - Lower RMSE% = better prediction accuracy
  - Edge years (2002, 2021) may show worse performance
    (fewer bracketing anchors for extrapolation)
  - 2011 (interior) typically shows best performance
    (well-bounded by 2002 and 2021 anchors)


## 4. Provenance Summary

For each pipeline, what fraction of gmina-year cells came from observed data
vs. estimated (interpolated/imputed)?

In [23]:
# ── Cell 17: Provenance summary ──

print("=" * 80)
print("PROVENANCE SUMMARY")
print("=" * 80)
for e_sid in all_e_sids:
    n = sum(1 for r in db._records.values()
            if e_sid in r.cross_tables and r.cross_tables[e_sid].years_with_data)
    if n == 0:
        continue
    prov_df = est.get_provenance_summary(e_sid)
    print(f"\n{e_sid}:")
    print(prov_df.to_string())

PROVENANCE SUMMARY

E_age_sex_1990:
Empty DataFrame
Columns: []
Index: []

E_age_sex_2000:
      n_units  mean_frac_observed  min_frac_observed
year                                                
1999     3069            0.811339                0.0
2000     3069            0.811339                0.0
2001     3069            0.811339                0.0
2002     3069            0.807755                0.0
2003     3069            0.807755                0.0
2004     3069            0.807755                0.0
2005     3069            0.807755                0.0
2006     3069            0.807755                0.0
2007     3069            0.807755                0.0
2008     3069            0.807755                0.0
2009     3069            0.807755                0.0
2010     3069            0.808081                0.0
2011     3069            0.808081                0.0
2012     3069            0.808081                0.0
2013     3069            0.808081                0.0
2014    

## 5. Overall Quality Report

Combine diagnostics, confidence, and LOOCV into a single quality assessment.

In [24]:
# ── Cell 18: Final quality report ──

print("="*80)
print("OVERALL ESTIMATION QUALITY REPORT")
print("="*80)
print()

# 1. Diagnostics summary
total_failures = sum(
    (d['status'] == 'FAIL').sum() for d in all_diag.values() if not d.empty
)
total_warnings = sum(
    (d['status'] == 'WARN').sum() for d in all_diag.values() if not d.empty
)
n_subjects_validated = len(all_diag)
print(f"1. CONSISTENCY DIAGNOSTICS")
print(f"   Subjects validated: {n_subjects_validated}")
print(f"   Total failures: {total_failures}")
print(f"   Total warnings: {total_warnings}")
if total_failures == 0:
    print(f"   → All consistency checks PASSED ✓")
else:
    # Breakdown failures by check type
    from collections import Counter
    check_counts = Counter()
    for d in all_diag.values():
        if not d.empty:
            fails = d[d['status'] == 'FAIL']
            for check in fails['check']:
                check_counts[check] += 1
    print(f"   Failure breakdown:")
    for check, count in check_counts.most_common():
        print(f"     {check}: {count}")
    print(f"   Note: marginal_consistency failures are all in E_educ*_1990 (ogółem residual)")
    print(f"         non_negativity: 6 cells in E_age_sex_2000 (minor numerical artifacts)")
print()

# 2. Confidence summary
print(f"2. ESTIMATION CONFIDENCE")
for e_sid, conf_df in all_confidence.items():
    avg = conf_df.groupby('teryt_id')['confidence'].mean()
    below_40 = (avg < 40).sum()
    print(f"   {e_sid:22s}: mean={avg.mean():.1f}, "
          f"{below_40:4d} gminas below 40 ({below_40/len(avg)*100:.1f}%)")
print()

# 3. LOOCV summary
print(f"3. CROSS-VALIDATION (age_sex_2000)")
for holdout, df in [('2002', loocv_2002_df), ('2011', loocv_df), ('2021', loocv_2021_df)]:
    if df is not None and not df.empty:
        print(f"   Holdout {holdout}: mean RMSE%={df['cell_rmse_pct'].mean():.1f}%, "
              f"median={df['cell_rmse_pct'].median():.1f}%, "
              f"n={len(df)} gminas")
print()

# 4. Provenance (age_sex_2000 only)
print(f"4. PROVENANCE (E_age_sex_2000)")
prov = est.get_provenance_summary('E_age_sex_2000')
if not prov.empty:
    mean_obs = prov['mean_frac_observed'].mean()
    print(f"   Mean fraction observed across years: {mean_obs:.1%}")
    print(f"   (80% observed = BDL P2137 annual data covers ~80% of gminas)")
else:
    print(f"   Provenance not available")
print()

# 5. Key findings
print(f"5. KEY FINDINGS")
print(f"   ✓ Non-negativity: 6 minor violations in E_age_sex_2000 (out of 64,486 cells)")
print(f"   ✓ Marginal consistency: perfect for 2000-era subjects; "
      f"2334 issues in 1990 educ (residual category)")
print(f"   ✓ Hierarchical consistency: PERFECT across all subjects (0 failures)")
print(f"   ✓ Population match: PERFECT for age_sex (0/88,302 failures)")
print(f"   ✓ LOOCV: median cell RMSE ~3.6% — excellent prediction accuracy")
print(f"   ✓ Temporal smoothness: warnings expected at TERYT reform (1998→1999) "
      f"and census boundaries")
print(f"   ⚠ E_educ_sex_1990 has lowest confidence (mean 37.2) — limited anchor data")
print(f"   ⚠ Warsaw districts have very low confidence (no census anchors)")

print()
print("="*80)
print("END OF QUALITY REPORT")
print("="*80)

OVERALL ESTIMATION QUALITY REPORT

1. CONSISTENCY DIAGNOSTICS
   Subjects validated: 8
   Total failures: 4674
   Total warnings: 35431
   Failure breakdown:
     marginal_consistency: 4668
     non_negativity: 6
   Note: marginal_consistency failures are all in E_educ*_1990 (ogółem residual)
         non_negativity: 6 cells in E_age_sex_2000 (minor numerical artifacts)

2. ESTIMATION CONFIDENCE
   E_age_sex_1990        : mean=47.4,   12 gminas below 40 (0.4%)
   E_age_sex_2000        : mean=84.6,   11 gminas below 40 (0.4%)
   E_educ_1990           : mean=52.6,   91 gminas below 40 (3.6%)
   E_educ_2000           : mean=50.0,  159 gminas below 40 (6.2%)
   E_educ_sex_1990       : mean=37.2, 2357 gminas below 40 (93.1%)
   E_educ_sex_2000       : mean=50.0,  159 gminas below 40 (6.2%)
   E_hh_size_1990        : mean=52.6,   91 gminas below 40 (3.6%)
   E_hh_size_2000        : mean=50.0,  159 gminas below 40 (6.2%)

3. CROSS-VALIDATION (age_sex_2000)
   Holdout 2002: mean RMSE%=6.3%, me

In [25]:
# ── Cell 19: Investigate marginal consistency failures in E_educ_1990 ──

diag = all_diag.get('E_educ_1990', pd.DataFrame())
marg = diag[diag['check'] == 'marginal_consistency']
print(f"E_educ_1990 marginal consistency failures: {len(marg)}")
print(f"Unique gminas: {marg['teryt_id'].nunique()}")
print(f"Year distribution:")
print(marg['year'].value_counts().sort_index().to_string())

# Pick a sample failure to understand what's happening
if not marg.empty:
    sample = marg.iloc[0]
    tid = sample['teryt_id']
    yr = sample['year']
    rec = db._records[tid]
    ct = rec.cross_tables['E_educ_1990']
    tbl = ct.tables[yr]
    print(f"\nSample: {tid} {rec.name} year={yr}")
    print(f"  Labels: {ct.dim_labels}")
    print(f"  Table: {tbl}")
    # Find ogółem
    labels = ct.dim_labels[ct.dim_names[0]]
    og_idx = None
    for i, lbl in enumerate(labels):
        if 'ogółem' in lbl.lower():
            og_idx = i
    if og_idx is not None:
        non_og = [i for i in range(len(labels)) if i != og_idx]
        og_val = tbl[og_idx]
        sub_sum = sum(tbl[i] for i in non_og)
        print(f"  ogółem (idx {og_idx}) = {og_val:.2f}")
        print(f"  sum of others = {sub_sum:.2f}")
        print(f"  diff = {og_val - sub_sum:.4f}")
        print(f"  Detail: {sample['detail']}")

E_educ_1990 marginal consistency failures: 2334
Unique gminas: 389
Year distribution:
year
1986    389
1987    389
1991    389
1992    389
1993    389
1994    389

Sample: 0200000 DOLNOŚLĄSKIE year=1986
  Labels: {'n1': ['ogółem', 'podstawowe', 'podstawowe nieukończone i bez wykształcenia', 'wyższe', 'zasadnicze zawodowe', 'średnie']}
  Table: [     0.         830363.27651101      0.          97592.69674643
 436388.9557879  461257.41339596]
  ogółem (idx 0) = 0.00
  sum of others = 1825602.34
  diff = -1825602.3424
  Detail: dim=n1 og=0.0 sum=1825602.3 diff=-1825602.3424


In [26]:
# ── Cell 20: Root cause — check gmina-level ogółem for E_educ_1990 ──

# Are the parent-level failures caused by gmina-level ogółem being wrong?
e_sid = 'E_educ_1990'
# Check a gmina in DOLNOŚLĄSKIE for 1986
sample_tids = [tid for tid, rec in db._records.items()
               if rec.level == LEVEL_GMINA and tid[-1] in {'1','2','3'}
               and tid.startswith('02') and e_sid in rec.cross_tables][:3]

for tid in sample_tids:
    rec = db._records[tid]
    ct = rec.cross_tables[e_sid]
    labels = ct.dim_labels[ct.dim_names[0]]
    print(f"\n{tid} {rec.name} (rodz={rec.rodz})")
    for yr in [1986, 1988, 1995, 2002]:
        tbl = ct.tables.get(yr)
        if tbl is None:
            print(f"  {yr}: no data")
        else:
            og_idx = next((i for i, l in enumerate(labels) if 'ogółem' in l.lower()), None)
            if og_idx is not None:
                non_og = [i for i in range(len(labels)) if i != og_idx]
                print(f"  {yr}: ogółem={tbl[og_idx]:.1f}, sum_others={sum(tbl[i] for i in non_og):.1f}, "
                      f"diff={tbl[og_idx]-sum(tbl[i] for i in non_og):.4f}")
            else:
                print(f"  {yr}: {tbl}")

# Check which levels the 389 failing records are at
marg = all_diag['E_educ_1990'][all_diag['E_educ_1990']['check'] == 'marginal_consistency']
level_counts = {}
for tid in marg['teryt_id'].unique():
    rec = db._records[tid]
    lev = rec.level
    level_counts[lev] = level_counts.get(lev, 0) + 1
print(f"\nFailing records by level: {level_counts}")
print(f"  (2=voivodeship, 5=powiat, 6=gmina)")


0201011 Bolesławiec (rodz=1)
  1986: ogółem=nan, sum_others=nan, diff=nan
  1988: ogółem=29403.6, sum_others=29403.6, diff=0.0000
  1995: ogółem=34135.7, sum_others=34135.7, diff=0.0000
  2002: ogółem=36152.0, sum_others=36152.0, diff=0.0000

0201022 Bolesławiec (rodz=2)
  1986: ogółem=nan, sum_others=nan, diff=nan
  1988: ogółem=6855.7, sum_others=6855.7, diff=0.0000
  1995: ogółem=8325.9, sum_others=8325.9, diff=0.0000
  2002: ogółem=9714.0, sum_others=9714.0, diff=0.0000

0201032 Gromadka (rodz=2)
  1986: ogółem=nan, sum_others=nan, diff=nan
  1988: ogółem=3915.2, sum_others=3915.2, diff=0.0000
  1995: ogółem=4427.6, sum_others=4427.6, diff=0.0000
  2002: ogółem=4671.0, sum_others=4671.0, diff=0.0000

Failing records by level: {2: 16, 5: 373}
  (2=voivodeship, 5=powiat, 6=gmina)


In [27]:
# ── Cell 21: Trace ogółem in 1986 at gmina level for E_educ_1990 ──

e_sid = 'E_educ_1990'
# Find gminas with data for 1986
gminas_with_1986 = []
gminas_with_og_zero = []
for tid, rec in db._records.items():
    if rec.level != LEVEL_GMINA or tid[-1] not in {'1','2','3'}:
        continue
    ct = rec.cross_tables.get(e_sid)
    if ct is None:
        continue
    tbl = ct.tables.get(1986)
    if tbl is None or np.all(np.isnan(tbl)):
        continue
    gminas_with_1986.append(tid)
    labels = ct.dim_labels[ct.dim_names[0]]
    og_idx = next((i for i, l in enumerate(labels) if 'ogółem' in l.lower()), None)
    if og_idx is not None and abs(tbl[og_idx]) < 1.0:
        non_og = [i for i in range(len(labels)) if i != og_idx]
        sub = sum(tbl[i] for i in non_og)
        if sub > 100:
            gminas_with_og_zero.append((tid, rec.name, tbl[og_idx], sub))

print(f"Gminas with any E_educ_1990 data for 1986: {len(gminas_with_1986)}")
print(f"Gminas with ogółem≈0 but sub>100 in 1986: {len(gminas_with_og_zero)}")

if gminas_with_og_zero:
    print(f"\nSample (first 5):")
    for tid, name, og, sub in gminas_with_og_zero[:5]:
        print(f"  {tid} {name:30s}  ogółem={og:.2f}  sum_others={sub:.1f}")
elif gminas_with_1986:
    print(f"\nAll gminas have correct ogółem for 1986!")
    # Show a sample to confirm
    for tid in gminas_with_1986[:3]:
        rec = db._records[tid]
        ct = rec.cross_tables[e_sid]
        tbl = ct.tables[1986]
        labels = ct.dim_labels[ct.dim_names[0]]
        og_idx = next((i for i, l in enumerate(labels) if 'ogółem' in l.lower()), None)
        non_og = [i for i in range(len(labels)) if i != og_idx]
        print(f"  {tid} {rec.name:30s}  ogółem={tbl[og_idx]:.1f}  "
              f"sum={sum(tbl[i] for i in non_og):.1f}")
else:
    print("No gminas have data for 1986 — the pipeline doesn't extrapolate to 1986?")

Gminas with any E_educ_1990 data for 1986: 2531
Gminas with ogółem≈0 but sub>100 in 1986: 0

All gminas have correct ogółem for 1986!
  0201011 Bolesławiec                     ogółem=nan  sum=nan
  0201022 Bolesławiec                     ogółem=nan  sum=nan
  0201032 Gromadka                        ogółem=nan  sum=nan


In [28]:
# ── Cell 22: Check gmina arrays directly for 1986 E_educ_1990 ──

e_sid = 'E_educ_1990'
# Find a gmina in DOLNOŚLĄSKIE with data for 1986
gmina_found = []
for tid, rec in db._records.items():
    if not tid.startswith('02') or rec.level != LEVEL_GMINA:
        continue
    if tid[-1] not in {'1','2','3'}:
        continue
    ct = rec.cross_tables.get(e_sid)
    if ct is None:
        continue
    tbl = ct.tables.get(1986)
    if tbl is not None:
        has_nonnan = not np.all(np.isnan(tbl))
        gmina_found.append((tid, rec.name, tbl.copy(), has_nonnan))

print(f"Gminas in DOLNOŚLĄSKIE with 1986 table: {len(gmina_found)}")
print(f"With at least one non-NaN: {sum(1 for _, _, _, h in gmina_found if h)}")

# Show first 5 arrays
for tid, name, tbl, has in gmina_found[:5]:
    print(f"\n  {tid} {name}")
    print(f"    array: {tbl}")
    print(f"    has_nonnan: {has}")

# Now check a powiat in DOLNOŚLĄSKIE
for tid, rec in db._records.items():
    if tid.startswith('0201') and rec.level == LEVEL_POWIAT:
        ct = rec.cross_tables.get(e_sid)
        if ct is not None:
            tbl = ct.tables.get(1986)
            print(f"\nPowiat {tid} {rec.name}")
            print(f"  1986: {tbl}")
            tbl88 = ct.tables.get(1988)
            print(f"  1988: {tbl88}")
        break

Gminas in DOLNOŚLĄSKIE with 1986 table: 170
With at least one non-NaN: 170

  0201011 Bolesławiec
    array: [           nan 11310.75489761            nan  1472.21615621
  6467.87537954  7816.74441649]
    has_nonnan: True

  0201022 Bolesławiec
    array: [          nan 3426.8530379            nan   55.88167968 1761.44309301
  704.51344526]
    has_nonnan: True

  0201032 Gromadka
    array: [          nan 1954.41692735           nan   47.89858258  866.84548878
  509.6140341 ]
    has_nonnan: True

  0201043 Nowogrodziec
    array: [          nan 4875.85244911           nan   93.13613279 2287.10080937
 1009.64285126]
    has_nonnan: True

  0201052 Osiecznica
    array: [          nan 1811.75875747           nan   40.58074357  700.33279602
  373.02469267]
    has_nonnan: True

Powiat 0201000 bolesławiecki
  1986: [    0.         25901.62871554     0.          1762.26868405
 13306.32312433 11060.5415834 ]
  1988: [57725.73420068 24016.06595255  4649.32089575  1972.97999526
 15385.40322

In [34]:
# ── Re-run affected pipelines after NaN-factor fix ──
import importlib
import demographic_estimator as de_mod
importlib.reload(de_mod)

# Re-init estimator with fixed code
est = de_mod.DemographicEstimator(db, verbose=True)

pipelines = [
    ('age_sex',   '2000'), ('age_sex',   '1990'),
    ('educ',      '2000'), ('educ',      '1990'),
    ('educ_sex',  '2000'), ('educ_sex',  '1990'),
    ('hh_size',   '2000'), ('hh_size',   '1990'),
]
for var_type, section in pipelines:
    try:
        est.run_pipeline(var_type, section)
    except NotImplementedError as e:
        print(f"  SKIP {var_type}_{section}: {e}")
print("Done.")


DemographicEstimator initialised  (Gurobi=YES, IPFN=YES)

  PIPELINE: age_sex / Prediction2000
  Output subject: E_age_sex_2000
  Source: M_age_sex  shape=(16, 3)
  Gminas total: 2671, with M_age_sex: 2671
  Layer 1: generating seeds (log-linear interpolation)…
    Seeds generated: 2671/2671 units (skipped 0)
    2000: 2490 obs + 0 est
    2005: 2479 obs + 0 est
    2010: 2480 obs + 0 est
    2015: 2479 obs + 0 est
    2020: 2478 obs + 0 est
    2025: 0 obs + 2478 est
  Aggregating to powiat and voivodeship levels…
    Aggregated: 10229 powiat-years, 432 voiv-years
  Summary: 64486 observed + 2478 estimated cell-years stored
  ✓  E_age_sex_2000 complete

  PIPELINE: age_sex / Prediction1990
  Output subject: E_age_sex_1990
  Source: M_age_sex  shape=(16, 3)
  Phase A: constructing 1988 gmina age×sex via IPF…
    1988 IPF: 2478 OK, 193 skipped
  Phase B: building seeds (log-linear interpolation)…
    Seeds: 2671 gminas
  Phase C: old voivodeship marginal scaling (1986–1994)…
    Scaled 

In [37]:
# ── Validate all subjects after NaN fix ──
import pandas as pd

# Find all E_ subjects from records' cross tables
e_subjects = set()
for rec in db._records.values():
    for sid in rec.cross_tables:
        if sid.startswith('E_'):
            e_subjects.add(sid)
e_subjects = sorted(e_subjects)
print(f"E_ subjects: {e_subjects}")

all_diags = []
for sid in e_subjects:
    df = est.validate_results(sid)
    df['subject'] = sid
    all_diags.append(df)
    n_fail = (df['status'] == 'FAIL').sum()
    n_warn = (df['status'] == 'WARN').sum()
    print(f"  {sid}: {len(df)} checks, {n_fail} FAIL, {n_warn} WARN")

diag_all = pd.concat(all_diags, ignore_index=True)
fails = diag_all[diag_all['status'] == 'FAIL']
print(f"\nTotal: {len(diag_all)} checks, "
      f"{len(fails)} FAIL, "
      f"{(diag_all['status']=='WARN').sum()} WARN")

if len(fails) > 0:
    print("\nFailure breakdown:")
    print(fails.groupby(['subject', 'check']).size().to_string())


E_ subjects: ['E_age_sex_1990', 'E_age_sex_2000', 'E_educ_1990', 'E_educ_2000', 'E_educ_sex_1990', 'E_educ_sex_2000', 'E_hh_size_1990', 'E_hh_size_2000']
  Validating E_age_sex_1990: 2671 records, shape=(16, 3)
    [1] Non-negativity: 0 failures
    [2] Marginal consistency: 0 failures (tol=1.0)
    [3] Hierarchical consistency: 0 failures / 0 checked
    [4] Population match: 0 failures / 23816 checked (tol=0.1%)
    [5] Temporal smoothness: 5463 warnings (threshold=20%)
    [6] Sub-division consistency: 0 failures / 0 checked
  Validation complete: 5463 issues found
  E_age_sex_1990: 5463 checks, 0 FAIL, 5463 WARN
  Validating E_age_sex_2000: 3069 records, shape=(16, 3)
    [1] Non-negativity: 6 failures
    [2] Marginal consistency: 0 failures (tol=1.0)
    [3] Hierarchical consistency: 0 failures / 10229 checked
    [4] Population match: 0 failures / 64486 checked (tol=0.1%)
    [5] Temporal smoothness: 7079 warnings (threshold=20%)
    [6] Sub-division consistency: 0 failures / 0 

In [38]:
# ── Inspect the 6 non-negativity failures ──
neg_fails = fails[fails['check'] == 'non_negativity']
print(neg_fails[['teryt_id', 'name', 'year', 'detail']].to_string())


     teryt_id        name  year                          detail
5463  2005022  Białowieża  2015   1 negative cells, min=-2.0000
5464  2005022  Białowieża  2016  1 negative cells, min=-17.0000
5465  2005022  Białowieża  2017  1 negative cells, min=-21.0000
5466  2005022  Białowieża  2018   1 negative cells, min=-2.0000
5467  2005022  Białowieża  2023  2 negative cells, min=-25.0000
5468  2005022  Białowieża  2024  2 negative cells, min=-48.0000


In [39]:
# Check source data and E_ table for Białowieża negatives
tid = '2005022'
rec = db._records[tid]

# Check M_age_sex source
ct = rec.cross_tables['M_age_sex']
for yr in [2015, 2016, 2017, 2023, 2024]:
    tbl = ct.tables.get(yr)
    if tbl is not None:
        neg = tbl[tbl < 0]
        print(f"M_age_sex {yr}: min={tbl.min():.1f}, negatives={len(neg)}")

# Check E_age_sex_2000
ect = rec.cross_tables['E_age_sex_2000']
for yr in [2015, 2016, 2017, 2023, 2024]:
    tbl = ect.tables.get(yr)
    if tbl is not None:
        neg_mask = tbl < 0
        if neg_mask.any():
            print(f"E_age_sex_2000 {yr}: negative cells:")
            labels_n1 = ect.dim_labels['n1']
            labels_n2 = ect.dim_labels['n2']
            for i in range(tbl.shape[0]):
                for j in range(tbl.shape[1]):
                    if tbl[i,j] < 0:
                        print(f"  [{labels_n1[i]}, {labels_n2[j]}] = {tbl[i,j]:.1f}")


M_age_sex 2015: min=-2.0, negatives=1
M_age_sex 2016: min=-17.0, negatives=1
M_age_sex 2017: min=-21.0, negatives=1
M_age_sex 2023: min=-25.0, negatives=2
M_age_sex 2024: min=-48.0, negatives=2
E_age_sex_2000 2015: negative cells:
  [20-24, mężczyźni] = -2.0
E_age_sex_2000 2016: negative cells:
  [20-24, mężczyźni] = -17.0
E_age_sex_2000 2017: negative cells:
  [20-24, mężczyźni] = -21.0
E_age_sex_2000 2023: negative cells:
  [20-24, mężczyźni] = -25.0
  [20-24, ogółem] = -7.0
E_age_sex_2000 2024: negative cells:
  [20-24, mężczyźni] = -48.0
  [20-24, ogółem] = -33.0


## Investigation Summary

### Bug Found & Fixed: NaN propagation in Layer 2 scaling

**Root cause**: Country-level `M_educ_1990` (from H_sex_educ) has NaN for "podstawowe nieukończone" in years 1986-87 and 1991-94 (H_sex_educ only has 4 education categories, not 5). During Layer 2 national scaling, `country_tbl / total_agg` produced NaN factors for that position, which propagated to gmina tables → ogółem also became NaN (sum includes NaN). After aggregation, NaN→0 via `np.nan_to_num`, causing 2,334 marginal consistency failures at powiat/voivodeship level.

**Fix**: Added `factors = np.where(np.isnan(factors), 1.0, factors)` in both `_estimate_educ_1990` and `_estimate_educ_sex_1990` Layer 2 scaling blocks. NaN in country data → no constraint for that category (keep seed estimate).

**Result**: All 2,334 marginal consistency failures eliminated. Only 6 non-negativity FAILs remain — from BDL source data in Białowieża (tiny gmina, ~2000 pop) where the 20-24 male age group has small negative values (-2 to -48) in the original BDL statistics.
